# RDFNet Resume Training from Epoch 116

This notebook resumes RDFNet training from a checkpoint saved at epoch 116.

## Setup Instructions:

### 1. Add Datasets in Kaggle (REQUIRED):
- **Checkpoint Dataset**: Your checkpoint containing `ep116-xxx.pth` file
  - Example path: `/kaggle/input/rdfnet-checkpoint-ep116`
- **Foggy VOC Dataset**: The foggy object detection training dataset
  - **MUST BE AT:** `/kaggle/input/foggy-voc/`
  - Search for "foggy voc" or "VOC_FOG_12K" in Kaggle datasets
  - Without this dataset, training will fail with 0 samples error

### 2. Enable GPU:
- Settings → Accelerator → GPU T4 x2

### 3. Update Configuration:
- Update `CHECKPOINT_INPUT_PATH` in Step 3 below with your checkpoint dataset path

### 4. Run All Cells

---

**Common Issues:**
- ❌ `ValueError: num_samples=0` → VOC dataset not added or wrong path
- ❌ `Checkpoint not found` → Update CHECKPOINT_INPUT_PATH in Step 3

## Step 1: Clone RDF_net Repository

In [ ]:
!rm -rf RDF_net
!git clone https://github.com/habibour/RDF_net.git
%cd RDF_net

## Step 2: Install Dependencies

In [ ]:
!pip install thop -q

## Step 3: Configuration - UPDATE THIS PATH!

In [ ]:
import os
import shutil

# ============= UPDATE THIS PATH =============
# Path to your checkpoint dataset in Kaggle
CHECKPOINT_INPUT_PATH = '/kaggle/input/rdfnet-checkpoint-ep116'  # UPDATE THIS!
# ============================================

# Working directory paths
CHECKPOINT_RESTORE_DIR = '/kaggle/working/checkpoints'
LOGS_DIR = '/kaggle/working/logs'

print("📍 Configuration:")
print(f"  Checkpoint input: {CHECKPOINT_INPUT_PATH}")
print(f"  Restore to: {CHECKPOINT_RESTORE_DIR}")

# Verify checkpoint exists
if os.path.exists(CHECKPOINT_INPUT_PATH):
    files = os.listdir(CHECKPOINT_INPUT_PATH)
    checkpoint_files = [f for f in files if f.endswith('.pth')]
    print(f"\n✅ Found checkpoint dataset")
    print(f"\n📦 Available checkpoints:")
    for f in checkpoint_files:
        print(f"  - {f}")
else:
    print(f"\n❌ ERROR: Checkpoint dataset not found!")
    print(f"\nPlease:")
    print(f"  1. Add your checkpoint dataset in Kaggle")
    print(f"  2. Update CHECKPOINT_INPUT_PATH above")
    raise FileNotFoundError(f"Checkpoint dataset not found: {CHECKPOINT_INPUT_PATH}")

## Step 4: Create VOC Classes File

In [ ]:
voc_classes = """aeroplane
bicycle
bird
boat
bottle
bus
car
cat
chair
cow
diningtable
dog
horse
motorbike
person
pottedplant
sheep
sofa
train
tvmonitor"""

with open('model_data/voc_classes.txt', 'w') as f:
    f.write(voc_classes)

print("✅ Created voc_classes.txt")

## Step 4.5: Verify Datasets Are Available

**CRITICAL CHECK:** Verifies both checkpoint and VOC datasets are properly added.
Without VOC dataset, training will fail with `num_samples=0` error.

In [ ]:
import os

print("🔍 Checking available Kaggle datasets...\n")

# List all input datasets
input_dir = '/kaggle/input'
if os.path.exists(input_dir):
    datasets = os.listdir(input_dir)
    print(f"📦 Available datasets in {input_dir}:")
    for ds in datasets:
        print(f"  - {ds}")
else:
    print("❌ /kaggle/input not found!")

print("\n" + "="*60)
print("🔍 Looking for Foggy VOC dataset...")
print("="*60)

# Check common VOC dataset paths
possible_paths = [
    '/kaggle/input/foggy-voc',
    '/kaggle/input/voc-fog',
    '/kaggle/input/foggy-object-detection',
    '/kaggle/input/voc2007-fog',
]

voc_found = False
voc_path = None

for path in possible_paths:
    if os.path.exists(path):
        voc_found = True
        voc_path = path
        print(f"\n✅ VOC dataset found at: {path}")
        print(f"\n📁 Dataset structure:")
        
        # Show directory structure
        for root, dirs, files in os.walk(path):
            level = root.replace(path, '').count(os.sep)
            if level < 3:  # Show 3 levels deep
                indent = '  ' * level
                print(f'{indent}{os.path.basename(root)}/')
                if level < 2:
                    sub_indent = '  ' * (level + 1)
                    for d in dirs[:5]:
                        print(f'{sub_indent}{d}/')
                    if len(dirs) > 5:
                        print(f'{sub_indent}... and {len(dirs) - 5} more directories')
        break

if not voc_found:
    print("\n❌ VOC dataset NOT FOUND!")
    print("\n⚠️ REQUIRED ACTION:")
    print("   1. Click 'Add Data' button in Kaggle")
    print("   2. Search for 'foggy voc' or 'VOC_FOG' dataset")
    print("   3. Add the dataset to your notebook")
    print("   4. Rerun this cell to verify")
    print("\n💡 Dataset should contain folders like:")
    print("   - VOC2007_FOG or VOC2007_Annotations")
    print("   - VOC2012_FOG or VOC2012_Annotations")
    print("\n⛔ Training cannot proceed without this dataset!")
else:
    print(f"\n✅ Dataset verified at: {voc_path}")

## Step 4.6: Auto-Fix VOC Dataset Paths in Training Script

**🔧 CRITICAL:** This cell automatically detects your VOC dataset location and updates the training script.
**You MUST run this before training or you'll get num_samples=0 error!**

In [ ]:
import os
import xml.etree.ElementTree as ET
import random

print("="*70)
print("📝 GENERATING ANNOTATION FILES")
print("="*70)

VOC_CLASSES = ["aeroplane", "bicycle", "bird", "boat", "bottle", "bus", "car", "cat", 
               "chair", "cow", "diningtable", "dog", "horse", "motorbike", "person", 
               "pottedplant", "sheep", "sofa", "train", "tvmonitor"]

# Find VOC dataset
print("\n🔍 Scanning for VOC dataset...\n")

voc_datasets = []

for root, dirs, files in os.walk('/kaggle/input'):
    for d in dirs:
        if 'VOC2007_FOG' in d or 'voc2007_fog' in d.lower():
            fog_path = os.path.join(root, d)
            # Try to find corresponding annotations
            parent_dir = os.path.dirname(fog_path)
            for ann_dir in os.listdir(parent_dir):
                if 'annotation' in ann_dir.lower() and 'voc2007' in ann_dir.lower():
                    ann_path = os.path.join(parent_dir, ann_dir)
                    voc_datasets.append((fog_path, ann_path, 'VOC2007'))
                    print(f"  ✅ Found VOC2007:")
                    print(f"     Images: {fog_path}")
                    print(f"     Annotations: {ann_path}")
                    break
        
        elif 'VOC2012_FOG' in d or 'voc2012_fog' in d.lower():
            fog_path = os.path.join(root, d)
            parent_dir = os.path.dirname(fog_path)
            for ann_dir in os.listdir(parent_dir):
                if 'annotation' in ann_dir.lower() and 'voc2012' in ann_dir.lower():
                    ann_path = os.path.join(parent_dir, ann_dir)
                    voc_datasets.append((fog_path, ann_path, 'VOC2012'))
                    print(f"  ✅ Found VOC2012:")
                    print(f"     Images: {fog_path}")
                    print(f"     Annotations: {ann_path}")
                    break

if not voc_datasets:
    print("\n❌ No VOC datasets found!")
    print("Please ensure the Foggy VOC dataset is added in Kaggle")
    raise FileNotFoundError("VOC dataset not found")

print(f"\n✅ Found {len(voc_datasets)} VOC dataset(s)")

# Generate annotations
print("\n" + "="*70)
print("📝 Processing images and annotations...")
print("="*70 + "\n")

all_lines = []

for fog_dir, ann_dir, name in voc_datasets:
    if not os.path.exists(fog_dir) or not os.path.exists(ann_dir):
        print(f"⚠️ Skipping {name}: paths not accessible")
        continue
    
    # Get all images
    images = [f for f in os.listdir(fog_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]
    print(f"📁 {name}: Processing {len(images)} images...")
    
    processed = 0
    for img_name in images:
        img_path = os.path.join(fog_dir, img_name)
        xml_name = os.path.splitext(img_name)[0] + '.xml'
        xml_path = os.path.join(ann_dir, xml_name)
        
        if not os.path.exists(xml_path):
            continue
        
        # Parse XML
        try:
            tree = ET.parse(xml_path)
            root = tree.getroot()
            
            boxes = []
            for obj in root.iter('object'):
                cls_name = obj.find('name').text
                if cls_name not in VOC_CLASSES:
                    continue
                
                cls_id = VOC_CLASSES.index(cls_name)
                bbox = obj.find('bndbox')
                box = [
                    int(float(bbox.find('xmin').text)),
                    int(float(bbox.find('ymin').text)),
                    int(float(bbox.find('xmax').text)),
                    int(float(bbox.find('ymax').text)),
                    cls_id
                ]
                boxes.append(','.join(map(str, box)))
            
            if boxes:
                line = img_path + ' ' + ' '.join(boxes)
                all_lines.append(line)
                processed += 1
        except Exception as e:
            continue
    
    print(f"  ✅ {name}: Processed {processed} images with annotations")

# Split into train/val
random.seed(114514)
random.shuffle(all_lines)
split_idx = int(len(all_lines) * 0.9)
train_lines = all_lines[:split_idx]
val_lines = all_lines[split_idx:]

# Write annotation files
train_path = '/kaggle/working/train_12k.txt'
val_path = '/kaggle/working/val_12k.txt'

with open(train_path, 'w') as f:
    f.write('\n'.join(train_lines))

with open(val_path, 'w') as f:
    f.write('\n'.join(val_lines))

print("\n" + "="*70)
print("✅ ANNOTATION FILES CREATED SUCCESSFULLY!")
print("="*70)
print(f"\n📊 Training samples: {len(train_lines)}")
print(f"📊 Validation samples: {len(val_lines)}")
print(f"\n📁 Files created:")
print(f"  - {train_path}")
print(f"  - {val_path}")
print("\n✅ Ready for training!")
print("="*70)

## Step 4.7: Generate Annotation Files Manually

**🔧 CRITICAL FIX:** This manually creates the annotation files needed for training.
This bypasses any path issues and ensures annotations are created correctly.

In [ ]:
import os
import re

print("="*70)
print("🔧 AUTO-FIXING VOC DATASET PATHS")
print("="*70)

print("\n🔍 Scanning for VOC dataset structure...\n")

# Find VOC dataset base
voc_base = None
for root, dirs, files in os.walk('/kaggle/input'):
    for d in dirs:
        if 'VOC2007' in d or 'VOC2012' in d:
            voc_base = root
            break
    if voc_base:
        break

if not voc_base:
    print("❌ VOC dataset not found!")
    print("\n⚠️ Please add the Foggy VOC dataset in Kaggle and rerun this cell")
    raise FileNotFoundError("VOC dataset not found in /kaggle/input")

print(f"✅ Found VOC dataset at: {voc_base}\n")

# Scan for all required folders
voc2007_fog = None
voc2007_ann = None
voc2012_fog = None
voc2012_ann = None

for root, dirs, files in os.walk(voc_base):
    for d in dirs:
        full_path = os.path.join(root, d)
        
        # Match VOC2007 fog images
        if ('VOC2007_FOG' in d or 'voc2007_fog' in d.lower() or 
            (d == 'VOC2007' and 'fog' in root.lower())):
            if not voc2007_fog:  # Take first match
                voc2007_fog = full_path
                print(f"  ✅ VOC2007 Fog Images: {full_path}")
        
        # Match VOC2007 annotations
        elif ('VOC2007_Annotations' in d or 'VOC2007_annotations' in d or 
              (d == 'Annotations' and 'VOC2007' in root and 'voc2012' not in root.lower())):
            if not voc2007_ann:
                voc2007_ann = full_path
                print(f"  ✅ VOC2007 Annotations: {full_path}")
        
        # Match VOC2012 fog images
        elif ('VOC2012_FOG' in d or 'voc2012_fog' in d.lower() or 
              (d == 'VOC2012' and 'fog' in root.lower())):
            if not voc2012_fog:
                voc2012_fog = full_path
                print(f"  ✅ VOC2012 Fog Images: {full_path}")
        
        # Match VOC2012 annotations
        elif ('VOC2012_Annotations' in d or 'VOC2012_annotations' in d or 
              (d == 'Annotations' and 'VOC2012' in root and 'voc2007' not in root.lower())):
            if not voc2012_ann:
                voc2012_ann = full_path
                print(f"  ✅ VOC2012 Annotations: {full_path}")

# Validate what we found
print("\n" + "="*70)
print("📊 Dataset Validation:")
print("="*70)

has_voc2007 = bool(voc2007_fog and voc2007_ann)
has_voc2012 = bool(voc2012_fog and voc2012_ann)

if has_voc2007:
    print("✅ VOC2007 dataset COMPLETE")
else:
    print(f"⚠️ VOC2007 incomplete - Images: {bool(voc2007_fog)}, Annotations: {bool(voc2007_ann)}")

if has_voc2012:
    print("✅ VOC2012 dataset COMPLETE")
else:
    print(f"⚠️ VOC2012 incomplete - Images: {bool(voc2012_fog)}, Annotations: {bool(voc2012_ann)}")

if not (has_voc2007 or has_voc2012):
    print("\n❌ ERROR: No complete VOC dataset found!")
    print("At least VOC2007 or VOC2012 must have both images and annotations")
    raise ValueError("Incomplete VOC dataset")

# Update kaggle_train.py
print("\n" + "="*70)
print("🔧 Updating kaggle_train.py...")
print("="*70)

train_script = 'kaggle_train.py'

if not os.path.exists(train_script):
    print(f"❌ {train_script} not found!")
    raise FileNotFoundError(f"{train_script} not found in current directory")

with open(train_script, 'r') as f:
    content = f.read()

# Replace all VOC path variables
old_patterns = {
    'VOC2007_FOG': r"VOC2007_FOG\s*=\s*['\"].*?['\"]",
    'VOC2007_ANN': r"VOC2007_ANN\s*=\s*['\"].*?['\"]",
    'VOC2012_FOG': r"VOC2012_FOG\s*=\s*['\"].*?['\"]",
    'VOC2012_ANN': r"VOC2012_ANN\s*=\s*['\"].*?['\"]",
}

replacements = {
    'VOC2007_FOG': f"VOC2007_FOG = '{voc2007_fog or ''}'",
    'VOC2007_ANN': f"VOC2007_ANN = '{voc2007_ann or ''}'",
    'VOC2012_FOG': f"VOC2012_FOG = '{voc2012_fog or ''}'",
    'VOC2012_ANN': f"VOC2012_ANN = '{voc2012_ann or ''}'",
}

for key, pattern in old_patterns.items():
    if re.search(pattern, content):
        content = re.sub(pattern, replacements[key], content)
        print(f"  ✅ Updated {key}")
    else:
        print(f"  ⚠️ Pattern not found: {key}")

# Write updated script
with open(train_script, 'w') as f:
    f.write(content)

print("\n" + "="*70)
print("✅ kaggle_train.py UPDATED SUCCESSFULLY!")
print("="*70)
print("\nUpdated paths:")
print(f"  VOC2007_FOG = '{voc2007_fog}'")
print(f"  VOC2007_ANN = '{voc2007_ann}'")
print(f"  VOC2012_FOG = '{voc2012_fog}'")
print(f"  VOC2012_ANN = '{voc2012_ann}'")
print("\n✅ Script is now configured for your dataset!")
print("\n➡️ You can now proceed to Step 5 to restore checkpoint")
print("="*70)

## Step 5: Restore Checkpoint to Working Directory

Copy your checkpoint from the input dataset to `/kaggle/working/checkpoints/` where the training script will find it.

In [ ]:
# Create checkpoint directory
os.makedirs(CHECKPOINT_RESTORE_DIR, exist_ok=True)
os.makedirs(LOGS_DIR, exist_ok=True)

# Copy checkpoint files from input dataset to working directory
if os.path.exists(CHECKPOINT_INPUT_PATH):
    print("📦 Restoring checkpoints...")
    
    # Get all .pth files
    checkpoint_files = [f for f in os.listdir(CHECKPOINT_INPUT_PATH) if f.endswith('.pth')]
    
    if not checkpoint_files:
        print("❌ No .pth files found in checkpoint dataset!")
        print("\n⚠️ Make sure you:")
        print("   1. Added your checkpoint dataset in Kaggle")
        print("   2. The .pth file is in the root of the dataset")
    else:
        for filename in checkpoint_files:
            src = os.path.join(CHECKPOINT_INPUT_PATH, filename)
            dst = os.path.join(CHECKPOINT_RESTORE_DIR, filename)
            shutil.copy2(src, dst)
            size_mb = os.path.getsize(dst) / (1024 * 1024)
            print(f"  ✅ Restored: {filename} ({size_mb:.1f} MB)")
        
        # Find the epoch number to resume from
        epoch_files = [f for f in checkpoint_files if f.startswith('ep')]
        if epoch_files:
            # Get the latest epoch
            epochs = []
            for f in epoch_files:
                try:
                    epoch_num = int(f.split('-')[0].replace('ep', ''))
                    epochs.append(epoch_num)
                except:
                    pass
            
            if epochs:
                latest_epoch = max(epochs)
                print(f"\n📍 Will resume from epoch: {latest_epoch}")
        
        print(f"\n✅ Checkpoint restoration complete!")
        print(f"📁 Checkpoints saved to: {CHECKPOINT_RESTORE_DIR}")
else:
    print("❌ Checkpoint input path not found!")
    print(f"⚠️ Looking for: {CHECKPOINT_INPUT_PATH}")
    print("\nPlease check:")
    print("   1. Is the checkpoint dataset added in Kaggle?")
    print("   2. Is the path correct in Step 3?")
    raise FileNotFoundError(f"Checkpoint dataset not found: {CHECKPOINT_INPUT_PATH}")

In [ ]:
import os

print("="*70)
print("🔍 CHECKPOINT DETECTION TEST")
print("="*70)

checkpoint_dir = '/kaggle/working/checkpoints'
save_dir = '/kaggle/working/logs'

print(f"\n📁 Checking: {checkpoint_dir}")

if not os.path.exists(checkpoint_dir):
    print(f"❌ Checkpoint directory not found!")
    print(f"⚠️ Please run Step 5 first to restore checkpoints")
else:
    files = os.listdir(checkpoint_dir)
    print(f"\n✅ Checkpoint directory exists")
    print(f"📦 Contents ({len(files)} files):")
    
    checkpoint_files = []
    for f in files:
        if f.endswith('.pth'):
            full_path = os.path.join(checkpoint_dir, f)
            size_mb = os.path.getsize(full_path) / (1024*1024)
            print(f"  - {f} ({size_mb:.1f} MB)")
            checkpoint_files.append(f)
    
    if not checkpoint_files:
        print("\n❌ No .pth files found in checkpoint directory!")
    else:
        # Find the epoch number
        epochs = []
        for f in checkpoint_files:
            if f.startswith('ep'):
                try:
                    epoch_num = int(f.split('-')[0].replace('ep', ''))
                    epochs.append((epoch_num, f))
                except:
                    pass
        
        if epochs:
            epochs.sort(reverse=True)
            latest_epoch, latest_file = epochs[0]
            
            print(f"\n✅ Latest checkpoint: {latest_file}")
            print(f"✅ Resume epoch: {latest_epoch}")
            
            # Now test if kaggle_train.py can find it
            print("\n" + "="*70)
            print("🔧 Testing kaggle_train.py checkpoint detection...")
            print("="*70)
            
            # Check what kaggle_train.py is looking for
            with open('kaggle_train.py', 'r') as f:
                content = f.read()
            
            if 'CHECKPOINT_DIR' in content and '/kaggle/working/checkpoints' in content:
                print("✅ kaggle_train.py is configured to look in /kaggle/working/checkpoints")
            else:
                print("⚠️ kaggle_train.py may not be looking in the right place")
                print("   It should search in /kaggle/working/checkpoints")
            
            # Also copy to logs directory as a backup
            logs_checkpoint = os.path.join(save_dir, latest_file)
            if not os.path.exists(logs_checkpoint):
                import shutil
                os.makedirs(save_dir, exist_ok=True)
                src = os.path.join(checkpoint_dir, latest_file)
                shutil.copy2(src, logs_checkpoint)
                print(f"\n💾 Also copied checkpoint to: {save_dir}")
                print(f"   (Backup location for training script)")
        else:
            print("\n⚠️ No epoch checkpoints found (files starting with 'ep')")
            print("   Found files:", checkpoint_files)

print("\n" + "="*70)
print("✅ Checkpoint verification complete")
print("="*70)

In [ ]:
import os
import shutil
import torch

print("="*70)
print("🔧 FIXING CHECKPOINT FILENAME")
print("="*70)

checkpoint_dir = '/kaggle/working/checkpoints'
old_name = 'latestTrained.pth'
old_path = os.path.join(checkpoint_dir, old_name)

if os.path.exists(old_path):
    print(f"\n✅ Found checkpoint: {old_name}")
    print(f"📦 Loading checkpoint to extract epoch info...")
    
    try:
        # Load checkpoint to get epoch number
        checkpoint = torch.load(old_path, map_location='cpu')
        
        # Try to extract epoch from checkpoint
        epoch = None
        if 'epoch' in checkpoint:
            epoch = checkpoint['epoch']
        elif 'start_epoch' in checkpoint:
            epoch = checkpoint['start_epoch']
        else:
            # If epoch not in checkpoint, assume it's epoch 116 based on user's request
            print("⚠️ Epoch info not found in checkpoint, using epoch 116")
            epoch = 116
        
        print(f"✅ Detected epoch: {epoch}")
        
        # Create new filename with proper format
        new_name = f'ep{epoch:03d}-val_loss0.0000.pth'
        new_path = os.path.join(checkpoint_dir, new_name)
        
        # Rename the file
        shutil.move(old_path, new_path)
        
        print(f"\n✅ Renamed checkpoint:")
        print(f"   From: {old_name}")
        print(f"   To:   {new_name}")
        
        # Also copy to logs directory
        logs_dir = '/kaggle/working/logs'
        os.makedirs(logs_dir, exist_ok=True)
        logs_checkpoint = os.path.join(logs_dir, new_name)
        shutil.copy2(new_path, logs_checkpoint)
        print(f"\n💾 Also copied to: {logs_dir}")
        
        print("\n" + "="*70)
        print("✅ CHECKPOINT READY FOR RESUME TRAINING!")
        print("="*70)
        print(f"\n📍 Training will resume from epoch: {epoch + 1}")
        print(f"📁 Checkpoint location: {new_path}")
        
    except Exception as e:
        print(f"\n❌ Error processing checkpoint: {e}")
        print("\n⚠️ Will manually rename to ep116 format...")
        new_name = 'ep116-val_loss0.0000.pth'
        new_path = os.path.join(checkpoint_dir, new_name)
        shutil.move(old_path, new_path)
        print(f"✅ Renamed to: {new_name}")
        
else:
    print(f"\n❌ Checkpoint not found: {old_path}")
    print("\n📁 Available files:")
    if os.path.exists(checkpoint_dir):
        for f in os.listdir(checkpoint_dir):
            print(f"  - {f}")
    else:
        print("  (checkpoint directory doesn't exist)")

print("\n" + "="*70)

In [ ]:
import torch
import os

print("="*70)
print("🔍 CHECKPOINT CONTENTS ANALYSIS")
print("="*70)

checkpoint_path = '/kaggle/working/checkpoints/ep116-val_loss0.0000.pth'

if os.path.exists(checkpoint_path):
    print(f"\n✅ Loading: {os.path.basename(checkpoint_path)}\n")
    
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    
    print("📦 Checkpoint structure:")
    print("="*70)
    
    if isinstance(checkpoint, dict):
        for key in checkpoint.keys():
            if isinstance(checkpoint[key], dict):
                print(f"\n✅ {key}: dict with {len(checkpoint[key])} items")
                # Show first few keys
                sample_keys = list(checkpoint[key].keys())[:3]
                for sk in sample_keys:
                    print(f"   - {sk}")
                if len(checkpoint[key]) > 3:
                    print(f"   ... and {len(checkpoint[key]) - 3} more")
            elif torch.is_tensor(checkpoint[key]):
                print(f"✅ {key}: tensor {tuple(checkpoint[key].shape)}")
            else:
                print(f"✅ {key}: {type(checkpoint[key]).__name__} = {checkpoint[key]}")
    else:
        print(f"⚠️ Checkpoint is not a dict, it's a {type(checkpoint).__name__}")
        print("   This means it only contains model weights")
    
    print("\n" + "="*70)
    print("📊 TRAINING STATE ASSESSMENT:")
    print("="*70)
    
    has_model = 'model' in checkpoint or 'state_dict' in checkpoint or isinstance(checkpoint, dict)
    has_optimizer = 'optimizer' in checkpoint if isinstance(checkpoint, dict) else False
    has_scheduler = 'scheduler' in checkpoint or 'lr_scheduler' in checkpoint if isinstance(checkpoint, dict) else False
    has_epoch = 'epoch' in checkpoint if isinstance(checkpoint, dict) else False
    has_ema = 'ema' in checkpoint if isinstance(checkpoint, dict) else False
    
    print(f"\n{'✅' if has_model else '❌'} Model weights: {has_model}")
    print(f"{'✅' if has_optimizer else '❌'} Optimizer state: {has_optimizer}")
    print(f"{'✅' if has_scheduler else '❌'} LR Scheduler: {has_scheduler}")
    print(f"{'✅' if has_epoch else '❌'} Epoch counter: {has_epoch}")
    print(f"{'✅' if has_ema else '❌'} EMA state: {has_ema}")
    
    print("\n" + "="*70)
    print("⚠️ TRAINING PIPELINE STATUS:")
    print("="*70)
    
    if has_model and has_optimizer and has_scheduler:
        print("\n✅ COMPLETE CHECKPOINT - Full resume capability")
        print("   Training will properly continue with:")
        print("   - Model weights from epoch 116")
        print("   - Optimizer state (momentum, learning rate)")
        print("   - Learning rate schedule")
        print("   ✅ This is the CORRECT way to resume training")
    elif has_model and not (has_optimizer or has_scheduler):
        print("\n⚠️ WEIGHTS-ONLY CHECKPOINT - Partial resume")
        print("   Training will resume with:")
        print("   ✅ Model weights from epoch 116")
        print("   ❌ Fresh optimizer (reset momentum)")
        print("   ❌ Reset learning rate (back to initial)")
        print("   ❌ Reset EMA")
        print("\n⚠️ THIS IS NOT IDEAL for resume training!")
        print("   The model knows what it learned, but optimizer doesn't.")
        print("   Training may be less stable initially.")
        print("\n💡 RECOMMENDATION:")
        print("   Training will still work, but:")
        print("   - Learning rate will restart from Init_lr")
        print("   - May see some instability for a few epochs")
        print("   - Will converge properly after adjustment period")
    else:
        print("\n❌ CHECKPOINT FORMAT UNKNOWN")
        print("   Please check checkpoint structure manually")
    
    print("\n" + "="*70)
    
else:
    print(f"\n❌ Checkpoint not found: {checkpoint_path}")
    print("\n📁 Run Step 5.6 first to rename the checkpoint")

## Step 5.7: Verify Checkpoint Contents

**🔍 IMPORTANT CHECK:** Verify what's saved in your checkpoint file.

## Step 5.6: Fix Checkpoint Filename

**🔧 CRITICAL FIX:** Renames `latestTrained.pth` to proper epoch format so training resumes correctly.

## Step 5.5: Verify Checkpoint Detection

**CRITICAL:** Ensures the training script can find and load your checkpoint.

## Step 6: Start Resume Training

The `kaggle_train.py` script will automatically:
- Detect the restored checkpoint in `/kaggle/working/checkpoints/`
- Generate annotation files from the VOC dataset
- Resume training from the saved epoch
- Continue training to the target epoch (default: 300)
- Save new checkpoints during training

**This will take several hours to complete.**

In [ ]:
!python kaggle_train.py

## Step 7: Monitor Training Progress

Run this cell periodically to check saved checkpoints and logs.

In [ ]:
import os

print("=" * 60)
print("📊 Training Progress")
print("=" * 60)

# List checkpoints
checkpoint_dir = '/kaggle/working/checkpoints'
if os.path.exists(checkpoint_dir):
    files = sorted(os.listdir(checkpoint_dir))
    print("\n💾 Saved checkpoints:")
    for f in files:
        if f.endswith('.pth'):
            size = os.path.getsize(os.path.join(checkpoint_dir, f)) / (1024*1024)
            print(f"  {f} ({size:.1f} MB)")
else:
    print("\n⚠️ No checkpoints directory found yet")

# Check logs
logs_dir = '/kaggle/working/logs'
if os.path.exists(logs_dir):
    log_folders = os.listdir(logs_dir)
    print(f"\n📁 Log folders: {len(log_folders)}")
    for folder in sorted(log_folders)[-3:]:  # Show last 3
        print(f"  - {folder}")
else:
    print("\n⚠️ No logs directory found yet")

print("\n" + "=" * 60)

## Step 8: Evaluate Model (Optional)

Run this after training completes to evaluate model performance.

In [ ]:
# Uncomment to run evaluation
# !python kaggle_eval.py

## Step 9: Prepare Files for Download

In [ ]:
import os

print("=" * 60)
print("📥 Files Ready for Download")
print("=" * 60)

# List all model files
output_files = []
for root, dirs, files in os.walk('/kaggle/working'):
    for f in files:
        if f.endswith('.pth'):
            file_path = os.path.join(root, f)
            size_mb = os.path.getsize(file_path) / (1024*1024)
            output_files.append((file_path, size_mb))

if output_files:
    print("\n💾 Model files to download:")
    for file_path, size_mb in sorted(output_files):
        print(f"  {file_path} ({size_mb:.1f} MB)")
    
    total_size = sum(size for _, size in output_files)
    print(f"\n📊 Total size: {total_size:.1f} MB")
    print("\n📥 Download from Kaggle Output tab")
else:
    print("\n⚠️ No model files found")

print("\n" + "=" * 60)

## Step 10: List All Output Files

In [ ]:
!echo "=== Working Directory ==="
!ls -lh /kaggle/working/

!echo ""
!echo "=== Logs Directory ==="
!ls -lh /kaggle/working/logs/ 2>/dev/null || echo "No logs yet"

!echo ""
!echo "=== Checkpoints Directory ==="
!ls -lh /kaggle/working/checkpoints/ 2>/dev/null || echo "No checkpoints yet"

---

## Summary & Download Instructions

### What This Notebook Does:
1. ✅ Clones the RDF_net repository from GitHub
2. ✅ Installs required dependencies (`thop`)
3. ✅ Verifies checkpoint and VOC datasets are properly added
4. ✅ Creates VOC classes file
5. ✅ Restores your epoch 116 checkpoint
6. ✅ Resumes training automatically from epoch 116
7. ✅ Saves new checkpoints every N epochs
8. ✅ Provides all files for download

### The Training Script Handles:
- ✅ Checkpoint detection and loading
- ✅ Annotation file generation from VOC dataset
- ✅ Resume epoch calculation
- ✅ Backbone freeze/unfreeze state
- ✅ Learning rate scheduling
- ✅ EMA updates
- ✅ Checkpoint saving

### Download Your Trained Models:
1. Wait for training to complete (several hours)
2. Go to Kaggle **Output** tab
3. Download the `/kaggle/working/checkpoints/` folder
4. Your new checkpoints will be there!

### Troubleshooting:
- **`num_samples=0` error** → VOC dataset not added or wrong path (check Step 4.5)
- **`Checkpoint not found`** → Update CHECKPOINT_INPUT_PATH in Step 3
- **Training not resuming** → Checkpoint not properly restored (check Step 5)